In [17]:
import sys, os
from pathlib import Path

PROJECT   = "/kaggle/working"
L2A_SRC   = "/kaggle/input/datasets/surathbob/hiwaf-l2a-v1"
L2B_SRC   = "/kaggle/input/datasets/surathbob/hiwaf-l2b-v1"
SPLIT_SRC = "/kaggle/input/datasets/surathbob/hiwaf-split-v1"
N06_SRC   = "/kaggle/input/datasets/surathbob/hiwaf-e2e-v1"   # wherever Notebook 06's results were pushed

sys.path.insert(0, PROJECT)
sys.path.insert(0, L2B_SRC)
sys.path.insert(0, SPLIT_SRC)
os.chdir(PROJECT)
for d in ["exported_models", "results"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"Working dir: {os.getcwd()}")

Working dir: /kaggle/working


In [18]:
%%capture
!pip install onnxruntime scipy pandas scikit-learn tqdm torch

In [19]:
import numpy as np
import pandas as pd
import json, re, hashlib, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.metrics import f1_score

import tensorflow as tf
from feature_engineering.extractor import extract_features, to_vector, FEATURE_NAMES, strip_to_path_query, INPUT_DIM, _SQLI, _XSS, _LFI, _OSCI
from feature_engineering.tokenizer import CharTokenizer
from feature_engineering.normalizer import Normalizer
from layer2b.candidates.bigru import BiGRUClassifier

assert INPUT_DIM == 29
CLASS_NAMES = ["normal", "sqli", "xss", "lfi", "other_attack"]
ATTACK_REGEX = {"sqli": _SQLI, "xss": _XSS, "lfi": _LFI, "other_attack": _OSCI}

# ── L2A: current state = post-Priority-D threshold, weights still original ─
d_report = json.load(open("/kaggle/input/datasets/surathbob/adaptive-retraining-output/results/07_adaptive_retraining_report.json"))  # adjust path if pushed elsewhere
L2A_THRESHOLD_CURRENT = d_report["l2a"]["threshold_after"]

autoencoder = tf.keras.models.load_model(f"{L2A_SRC}/exported_models/shallow_autoencoder.keras")
norm = Normalizer.load(f"{SPLIT_SRC}/exported_models/scaler_l2a.pkl")

# ── L2B: current state = Priority D's fine-tuned checkpoint ────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load("/kaggle/input/datasets/surathbob/adaptive-retraining-output/exported_models/layer2b_bigru_checkpoint_post_D.pt", map_location=device)
net = BiGRUClassifier(
    vocab_size=ckpt["vocab_size"], embed_dim=ckpt["train_params"]["embed_dim"],
    hidden_dim=ckpt["train_params"]["hidden_dim"], num_layers=ckpt["train_params"]["num_layers"],
    num_classes=ckpt["train_params"]["num_classes"],
).to(device)
net.load_state_dict(ckpt["state_dict"])
tok = CharTokenizer(max_len=512)

print(f"L2A current threshold (post-D): {L2A_THRESHOLD_CURRENT:.6f}")
print(f"L2B current checkpoint: post-D, val_f1={ckpt['provenance']['val_f1_after']:.4f}")

[Normalizer] Loaded from /kaggle/input/datasets/surathbob/hiwaf-split-v1/exported_models/scaler_l2a.pkl
L2A current threshold (post-D): 0.002985
L2B current checkpoint: post-D, val_f1=0.9931


In [20]:
traffic_pool = pd.read_csv(f"{N06_SRC}/results/06_pipeline_results.csv")
raw_text     = pd.read_csv(f"{SPLIT_SRC}/data/splits/l2b_test_raw.csv")

assert len(traffic_pool) == len(raw_text), (
    f"Row count mismatch: 06_pipeline_results.csv has {len(traffic_pool)}, "
    f"l2b_test_raw.csv has {len(raw_text)} — cannot safely merge by position."
)

traffic_pool = traffic_pool.reset_index(drop=True)
raw_text     = raw_text.reset_index(drop=True)
traffic_pool["url"]  = raw_text["url"]
traffic_pool["method"] = raw_text["method"]
traffic_pool["body"] = raw_text["body"]

# Verify the merge actually aligned correctly before trusting it — same
# check pattern used for the Notebook 02 split addendum.
assert (traffic_pool["true_label"] == raw_text["attack_class"]).all(), (
    "Merged url/body does not align with true_label — positional merge is WRONG, "
    "do not proceed. Check whether Notebook 06 dropped any rows to errors."
)
print("✅ Verified: url/body correctly aligned with true_label across all rows")

print(f"Simulated traffic pool: {len(traffic_pool):,} requests")
print(traffic_pool["decision"].value_counts())

✅ Verified: url/body correctly aligned with true_label across all rows
Simulated traffic pool: 3,693 requests
decision
block    2155
allow    1132
log       406
Name: count, dtype: int64


In [21]:
HEALTH_ERROR_THRESHOLD = 0.10
BASELINE_ERROR_RATE    = 0.02
SIMULATED_ERROR_RATE   = 0.15

def check_health_breach(error_rate: float, threshold: float = HEALTH_ERROR_THRESHOLD) -> bool:
    return error_rate > threshold

print("=== Server Health Monitor ===")
for label, rate in [("baseline", BASELINE_ERROR_RATE), ("simulated spike", SIMULATED_ERROR_RATE)]:
    breach = check_health_breach(rate)
    print(f"  {label:<16}: error_rate={rate:.1%}  threshold={HEALTH_ERROR_THRESHOLD:.1%}  -> breach={breach}")

CURRENT_ERROR_RATE = SIMULATED_ERROR_RATE
health_breach = check_health_breach(CURRENT_ERROR_RATE)

=== Server Health Monitor ===
  baseline        : error_rate=2.0%  threshold=10.0%  -> breach=False
  simulated spike : error_rate=15.0%  threshold=10.0%  -> breach=True


In [22]:
print("=== Health Breach Detected ===")
print(f"  Status: {'BREACH' if health_breach else 'NORMAL'}")
print(f"  Error rate: {CURRENT_ERROR_RATE:.1%} (threshold: {HEALTH_ERROR_THRESHOLD:.1%})")
print(f"  Action: {'Trigger feedback loop, capture recent passed traffic' if health_breach else 'No action'}")
assert health_breach, "No breach detected — nothing downstream should run in a real system"

=== Health Breach Detected ===
  Status: BREACH
  Error rate: 15.0% (threshold: 10.0%)
  Action: Trigger feedback loop, capture recent passed traffic


In [23]:
CAPTURE_PERCENTAGE = 100.0

# "Passed to the backend" = allow or log, per your architecture note —
# never block (those never reached the server).
eligible = traffic_pool[traffic_pool["decision"].isin(["allow", "log"])].copy()

rng = np.random.RandomState(42)
n_capture = int(len(eligible) * (CAPTURE_PERCENTAGE / 100.0))
captured_idx = rng.choice(eligible.index, size=n_capture, replace=False)
df_captured = eligible.loc[captured_idx].copy()

print(f"Eligible (allow/log) traffic in breach window: {len(eligible):,}")
print(f"Capture percentage: {CAPTURE_PERCENTAGE}%")
print(f"Captured: {len(df_captured):,} requests")

Eligible (allow/log) traffic in breach window: 1,538
Capture percentage: 100.0%
Captured: 1,538 requests


In [24]:
L2A_SCORE_MULTIPLIER, L2B_CONF_MULTIPLIER = 15.0, 50.0
LOG_THRESHOLD, BLOCK_THRESHOLD = 30, 70

def reaudit_decision(url, body):
    """Re-score using CURRENT L2A threshold + CURRENT (post-D) L2B."""
    req = {"url": url, "method": "GET", "headers": {}, "body": str(body) if pd.notna(body) else ""}
    fvec = norm.transform(to_vector(extract_features(req))).astype(np.float32)
    recon = autoencoder.predict(fvec, verbose=0)
    l2a_score = float(np.mean((fvec - recon) ** 2))

    if l2a_score < L2A_THRESHOLD_CURRENT:
        return "allow", "normal", 0.0, l2a_score

    text = f"GET {strip_to_path_query(url)} {req['body']}"
    tokens = tok.encode(text, pad=True).reshape(1, -1)
    net.eval()
    with torch.no_grad():
        logits = net(torch.from_numpy(tokens).long().to(device))
        proba = F.softmax(logits, dim=1).cpu().numpy()[0]
    pred_cls, conf = int(np.argmax(proba)), float(np.max(proba))
    pred_label = CLASS_NAMES[pred_cls]

    l2a_contrib = min(50.0, l2a_score * L2A_SCORE_MULTIPLIER)
    l2b_contrib = 0.0 if pred_cls == 0 else conf * L2B_CONF_MULTIPLIER
    score = min(100, int(l2a_contrib + l2b_contrib))
    decision = "block" if score >= BLOCK_THRESHOLD else "log" if score >= LOG_THRESHOLD else "allow"
    return decision, pred_label, conf, l2a_score

reaudit_results = df_captured.apply(lambda r: reaudit_decision(r["url"], r.get("body", "")), axis=1)
df_captured["reaudit_decision"]  = [r[0] for r in reaudit_results]
df_captured["reaudit_pred_label"] = [r[1] for r in reaudit_results]
df_captured["reaudit_confidence"] = [r[2] for r in reaudit_results]
df_captured["reaudit_l2a_score"]  = [r[3] for r in reaudit_results]

# Feedback candidate = original decision was allow/log AND re-audit disagrees
# (re-audit flags it as anomalous/attack) — NOT an arbitrary score threshold.
df_captured["disagreement"] = (
    df_captured["decision"].isin(["allow", "log"]) &
    (df_captured["reaudit_pred_label"] != "normal")
)

feedback_candidates = df_captured[df_captured["disagreement"]].copy()
print(f"Captured: {len(df_captured):,}")
print(f"Re-audit disagreements (feedback candidates): {len(feedback_candidates):,}")
print(f"\nOriginal decision vs re-audit label, among disagreements:")
print(feedback_candidates[["decision", "reaudit_decision", "reaudit_pred_label"]].value_counts())

Captured: 1,538
Re-audit disagreements (feedback candidates): 923

Original decision vs re-audit label, among disagreements:
decision  reaudit_decision  reaudit_pred_label
allow     log               lfi                   360
log       log               other_attack          220
allow     log               other_attack          158
log       log               sqli                  117
                            lfi                    31
                            xss                    29
allow     log               xss                     3
          allow             xss                     2
log       block             other_attack            1
allow     log               sqli                    1
log       block             lfi                     1
Name: count, dtype: int64


In [25]:
# ── Cell 7 (final) — Priority E measurable outputs ──────────────────────────
# Anti-Poison, Human Review, L2A recalibration, and L2B fine-tuning are NOT
# re-run here — that mechanism is already validated end-to-end in Notebook
# 07 (Priority D), including a proven poison-catch and an enforced approval
# gate. Re-running the same checks against a second dataset would validate
# the same mechanism twice without adding new evidence. Priority E's job is
# narrower and different: prove the health-breach trigger correctly
# identifies and captures real feedback-worthy traffic in the first place —
# the input side of the loop, not the loop itself.

print("=== PRIORITY E — HEALTH BREACH TRAFFIC CAPTURE ===")
print(f"Health breach detected: {'YES' if health_breach else 'NO'}")
print(f"Traffic window captured: {len(df_captured)} requests")
print(f"Feedback candidates (re-audit disagreements): {len(feedback_candidates)}")
print(f"  of which re-audit prediction matches ground truth: "
      f"{(feedback_candidates['true_label'] == feedback_candidates['reaudit_pred_label']).sum()}/{len(feedback_candidates)}")
print(f"Anti-poison / Human review / L2A / L2B update: deferred to the "
      f"already-validated Notebook 07 pipeline (same mechanism, not re-run here)")

final_report_e = {
    "health_breach_detected": bool(health_breach),
    "error_rate": CURRENT_ERROR_RATE, "error_threshold": HEALTH_ERROR_THRESHOLD,
    "traffic_window_captured": len(df_captured),
    "capture_percentage": CAPTURE_PERCENTAGE,
    "feedback_candidates": len(feedback_candidates),
    "feedback_candidate_precision_vs_ground_truth": float(
        (feedback_candidates["true_label"] == feedback_candidates["reaudit_pred_label"]).mean()
    ),
    "downstream_processing": "Anti-Poison / Human Review / L2A recalibration / L2B fine-tuning "
                              "validated separately in Notebook 07 (Priority D) — same shared "
                              "mechanism, intentionally not duplicated here.",
}
with open("results/08_health_feedback_report.json", "w") as f:
    json.dump(final_report_e, f, indent=2)
print(json.dumps(final_report_e, indent=2))

=== PRIORITY E — HEALTH BREACH TRAFFIC CAPTURE ===
Health breach detected: YES
Traffic window captured: 1538 requests
Feedback candidates (re-audit disagreements): 923
  of which re-audit prediction matches ground truth: 917/923
Anti-poison / Human review / L2A / L2B update: deferred to the already-validated Notebook 07 pipeline (same mechanism, not re-run here)
{
  "health_breach_detected": true,
  "error_rate": 0.15,
  "error_threshold": 0.1,
  "traffic_window_captured": 1538,
  "capture_percentage": 100.0,
  "feedback_candidates": 923,
  "feedback_candidate_precision_vs_ground_truth": 0.9934994582881906,
  "downstream_processing": "Anti-Poison / Human Review / L2A recalibration / L2B fine-tuning validated separately in Notebook 07 (Priority D) \u2014 same shared mechanism, intentionally not duplicated here."
}
